# NIBFS Fold-Fitted 1×5 — All Comparators

Notebook ini menjalankan perbandingan **apple-to-apple** antara:

- NIBFS;
- DEG-only;
- mRMR;
- LASSO.

Keempat metode menggunakan:

- 608 development samples yang sama;
- fold 1–5 yang sama;
- quantile normalization yang di-fit pada training fold;
- label-free ComBat yang di-fit pada training fold;
- bottom-10% variance filtering yang di-fit pada training fold;
- classifier LR, RF, dan LightGBM yang sama.

Implementasi feature selection diambil langsung dari modul `src/` repository ini,
sehingga comparator tidak dibuat ulang secara perkiraan.

Tidak ikut fitting:

- 152 locked internal holdout;
- GSE15852;
- GSE70947;
- TCGA-BRCA RNA-seq.

Klik **Runtime → Run all**.


In [ ]:

from __future__ import annotations
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, re, shutil, subprocess, sys, time

from google.colab import drive

DISCOVERY_GSE = [
    "GSE61304","GSE42568","GSE29044","GSE3744","GSE29431",
    "GSE26910","GSE31138","GSE71053","GSE10780","GSE30010",
    "GSE111662",
]
EXTERNAL_GEO = ["GSE15852", "GSE70947"]
EXTERNAL_RNASEQ = ["TCGA-BRCA"]

N_FOLDS = 5
FINAL_K = 20
RANDOM_STATE = 42
LOG2_THRESHOLD = 100.0
BOTTOM_VARIANCE_FRACTION = 0.10
AUC_MARGIN = 0.02
STABILITY_MARGIN = 0.10

assert set(DISCOVERY_GSE).isdisjoint(EXTERNAL_GEO)

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive", force_remount=False)

markers = list(Path("/content/drive/MyDrive").rglob("NIBFS_REPRODUCIBILITY_PACKAGE.marker"))
if len(markers) != 1:
    raise RuntimeError(f"Expected exactly one repository marker, found {len(markers)}: {markers}")
PACKAGE_DIR = markers[0].parent.resolve()
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
print("Repository:", PACKAGE_DIR)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "lightgbm", "statsmodels", "scipy", "scikit-learn",
    "matplotlib", "pyyaml", "git+https://github.com/Jfortin1/neuroCombat.git",
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import rankdata
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef,
    recall_score, precision_score, confusion_matrix, brier_score_loss,
)
from lightgbm import LGBMClassifier
from neuroCombat import neuroCombat, neuroCombatFromTraining

LOCAL_OUTPUT_DIR = (
    PACKAGE_DIR / "results" / "FOLD_FITTED_ALL_COMPARATORS_FROM_SERIES_MATRIX_1X5"
)
if LOCAL_OUTPUT_DIR.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIR)

TABLE_DIR = LOCAL_OUTPUT_DIR / "tables"
FIGURE_DIR = LOCAL_OUTPUT_DIR / "figures"
FOLD_DIR = LOCAL_OUTPUT_DIR / "folds"
TEMP_DIR = Path("/content/FFH_TEMP")

for folder in [TABLE_DIR, FIGURE_DIR, FOLD_DIR, TEMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def normalize_name(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())

def find_column(frame, candidates):
    lookup = {normalize_name(c): str(c) for c in frame.columns}
    for candidate in candidates:
        if normalize_name(candidate) in lookup:
            return lookup[normalize_name(candidate)]
    return None

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def clean_gene_symbol(value):
    text = str(value).strip()
    if not text or text.casefold() in {"nan","none","---"}:
        return ""
    for part in re.split(r"\s*///\s*|\s*//\s*|[;,|]", text):
        symbol = part.strip().upper()
        if re.fullmatch(r"[A-Z0-9][A-Z0-9._-]*", symbol):
            return symbol
    return ""

def detect_run_root(tables_dir):
    candidate = Path(tables_dir)
    while candidate != candidate.parent:
        if (
            (candidate / "raw" / "geo").is_dir()
            and (candidate / "results" / "main" / "tables").is_dir()
        ):
            return candidate
        candidate = candidate.parent
    raise RuntimeError(f"Cannot infer run root from {tables_dir}")

# R + limma
if shutil.which("Rscript") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "r-base"])

limma_ok = subprocess.run(
    ["Rscript", "-e",
     'quit(status=ifelse(requireNamespace("limma", quietly=TRUE),0,1))'],
    capture_output=True,
).returncode == 0

if not limma_ok:
    install_script = "\n".join([
        'options(repos = c(CRAN = "https://cloud.r-project.org"))',
        'if (!requireNamespace("BiocManager", quietly = TRUE)) {',
        '  install.packages("BiocManager", quiet = TRUE)',
        '}',
        'BiocManager::install("limma", ask = FALSE, update = FALSE, quiet = TRUE)',
    ])
    subprocess.check_call(["Rscript", "-e", install_script])

print("Setup complete.")


In [ ]:

import zipfile

MY_DRIVE = Path("/content/drive/MyDrive")

# -------------------------------------------------------------------------
# REPOSITORY SOURCE RESOLUTION
# Use the latest completed core run produced by notebooks/01_main_NIBFS_core.ipynb.
# -------------------------------------------------------------------------
run_candidates = []
for candidate in sorted((PACKAGE_DIR / "runs").glob("NIBFS_RAW_RUN_*")):
    tables = candidate / "results" / "main" / "tables"
    geo = candidate / "data" / "raw" / "geo"
    if (
        tables.is_dir()
        and geo.is_dir()
        and (tables / "cross_validated_predictions_all_k.csv").is_file()
        and (tables / "GPL570_probe_to_HGNC_mapping.csv").is_file()
        and (tables / "cross_validated_performance_summary_k20.csv").is_file()
    ):
        run_candidates.append(candidate)

if not run_candidates:
    raise FileNotFoundError(
        "No completed core run was found under PACKAGE_DIR/runs. "
        "Run notebooks/01_main_NIBFS_core.ipynb first."
    )

RUN_ROOT = max(run_candidates, key=lambda path: path.stat().st_mtime)
GEO_DIR = RUN_ROOT / "data" / "raw" / "geo"
TABLES_SOURCE_DIR = RUN_ROOT / "results" / "main" / "tables"
SOURCE_MODE = "latest completed core run in this repository"

series_paths = {
    gse: GEO_DIR / f"{gse}_series_matrix.txt.gz"
    for gse in DISCOVERY_GSE
}

missing_after_copy = [
    str(path)
    for path in series_paths.values()
    if not path.is_file()
]

if missing_after_copy:
    raise FileNotFoundError(
        "File discovery masih tidak lengkap setelah penyalinan:\n"
        + "\n".join(missing_after_copy)
    )

mandatory_after_copy = [
    TABLES_SOURCE_DIR / "cross_validated_predictions_all_k.csv",
    TABLES_SOURCE_DIR / "GPL570_probe_to_HGNC_mapping.csv",
    TABLES_SOURCE_DIR / "cross_validated_performance_summary_k20.csv",
]

missing_mandatory_after_copy = [
    str(path)
    for path in mandatory_after_copy
    if not path.is_file()
]

if missing_mandatory_after_copy:
    raise FileNotFoundError(
        "Tabel hasil masih tidak lengkap setelah penyalinan:\n"
        + "\n".join(missing_mandatory_after_copy)
    )

print("=" * 88)
print("SOURCE RESOLUTION: PASS")
print("=" * 88)
print("Source mode :", SOURCE_MODE)
print("Run root    :", RUN_ROOT)
print("GEO dir     :", GEO_DIR)
print("Tables      :", TABLES_SOURCE_DIR)
print("Discovery series matrices:", len(series_paths), "/ 11")
print(
    "Required table check:",
    sum(path.is_file() for path in mandatory_after_copy),
    "/ 3",
)


# Lock the original 608 development samples, labels, and folds


cv = pd.read_csv(
    TABLES_SOURCE_DIR / "cross_validated_predictions_all_k.csv"
)
method_col = find_column(cv, ["Feature_selection_method","Method"])
classifier_col = find_column(cv, ["Classifier","Model"])
k_col = find_column(cv, ["k","Panel_size"])
sample_col = find_column(cv, ["Sample_ID","GSM_ID","Sample"])
label_col = find_column(cv, ["True_Label","Label","y"])
fold_col = find_column(cv, ["Fold","CV_fold"])

locked = cv[
    cv[method_col].astype(str).str.casefold().eq("nibfs")
    & cv[classifier_col].astype(str).str.casefold().eq("lr")
    & pd.to_numeric(cv[k_col], errors="coerce").eq(FINAL_K)
][[sample_col,label_col,fold_col]].drop_duplicates(sample_col)

locked.columns = ["GSM_ID","Label","Fold"]
locked["GSM_ID"] = locked["GSM_ID"].astype(str).str.strip()
locked["Label"] = pd.to_numeric(locked["Label"]).astype(int)
locked["Fold"] = pd.to_numeric(locked["Fold"]).astype(int)
locked = locked.sort_values(["Fold","GSM_ID"]).reset_index(drop=True)

assert len(locked) == 608
assert sorted(locked["Fold"].unique().tolist()) == [1,2,3,4,5]

locked.to_csv(
    TABLE_DIR / "fold_fitted_1x5_fold_assignments.csv",
    index=False,
)

# Probe mapping
probe_map_raw = pd.read_csv(
    TABLES_SOURCE_DIR / "GPL570_probe_to_HGNC_mapping.csv"
)
probe_col = find_column(
    probe_map_raw,
    ["ID_REF","Probe_ID","Probe_Set_ID","Probe",
     "Affymetrix_Probe_Set_ID"],
)
gene_col = find_column(
    probe_map_raw,
    ["Gene","Gene_Symbol","HGNC_Symbol","Symbol","gene_symbol"],
)
if probe_col is None or gene_col is None:
    raise KeyError(
        f"Cannot detect mapping columns: {list(probe_map_raw.columns)}"
    )

probe_map = probe_map_raw[[probe_col,gene_col]].copy()
probe_map.columns = ["Probe","Gene"]
probe_map["Probe"] = probe_map["Probe"].astype(str).str.strip()
probe_map["Gene"] = probe_map["Gene"].map(clean_gene_symbol)
probe_map = (
    probe_map[
        probe_map["Probe"].ne("") & probe_map["Gene"].ne("")
    ]
    .sort_values(["Probe","Gene"])
    .drop_duplicates("Probe", keep="first")
)
probe_to_gene = probe_map.set_index("Probe")["Gene"].to_dict()

def read_series_matrix(path):
    frame = pd.read_csv(
        path, sep="\t", compression="gzip",
        comment="!", low_memory=False,
    )
    frame.columns = [
        str(c).strip().strip('"') for c in frame.columns
    ]
    id_col = find_column(frame, ["ID_REF","ID","Probe","Probe_ID"])
    if id_col is None:
        raise KeyError(f"No probe ID column in {path.name}")
    frame[id_col] = (
        frame[id_col].astype(str).str.strip().str.strip('"')
    )
    return (
        frame.rename(columns={id_col:"ID_REF"})
        .drop_duplicates("ID_REF", keep="first")
    )

development_ids = set(locked["GSM_ID"])
mapped_by_cohort = {}
sample_to_cohort = {}
log2_rows = []
selected_probe_rows = []

for gse in DISCOVERY_GSE:
    print("Mapping:", gse)
    raw = read_series_matrix(series_paths[gse])

    sample_columns = [
        c for c in raw.columns
        if c != "ID_REF" and str(c) in development_ids
    ]
    if not sample_columns:
        raise RuntimeError(f"No development samples found in {gse}")

    numeric = raw[sample_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    if numeric.isna().any().any():
        raise RuntimeError(f"Missing/non-numeric values in {gse}")

    values = numeric.to_numpy(dtype=float)
    apply_log2 = bool(
        np.min(values) >= 0 and np.max(values) > LOG2_THRESHOLD
    )
    if apply_log2:
        numeric = np.log2(numeric + 1.0)

    numeric.index = raw["ID_REF"].astype(str)
    available = numeric.index.intersection(
        pd.Index(probe_to_gene.keys())
    )
    numeric = numeric.loc[available]
    variances = numeric.var(axis=1, ddof=1)

    selection = pd.DataFrame({
        "Probe": numeric.index,
        "Gene": [probe_to_gene[p] for p in numeric.index],
        "Variance_all_development_within_cohort":
            variances.to_numpy(),
    }).sort_values(
        ["Gene","Variance_all_development_within_cohort","Probe"],
        ascending=[True,False,True],
    ).drop_duplicates("Gene", keep="first")

    selected_probe_rows.extend(
        selection.assign(GEO_ID=gse).to_dict("records")
    )

    chosen = numeric.loc[selection["Probe"].tolist()].copy()
    chosen.index = selection["Gene"].tolist()
    gene_matrix = chosen.T
    mapped_by_cohort[gse] = gene_matrix

    for sample_id in gene_matrix.index.astype(str):
        if sample_id in sample_to_cohort:
            raise RuntimeError(f"Duplicate sample across cohorts: {sample_id}")
        sample_to_cohort[sample_id] = gse

    q = np.percentile(values.ravel(), [0,1,25,50,75,99,100])
    log2_rows.append({
        "GEO_ID":gse,
        "Development_samples":len(sample_columns),
        "Probe_rows_original":len(raw),
        "Probe_rows_mapped":len(numeric),
        "Genes_after_representative_probe":gene_matrix.shape[1],
        "Minimum":q[0],"Q01":q[1],"Q25":q[2],"Median":q[3],
        "Q75":q[4],"Q99":q[5],"Maximum":q[6],
        "Log2_threshold":LOG2_THRESHOLD,
        "Log2_applied":apply_log2,
    })

common_genes = sorted(
    set.intersection(
        *[set(x.columns) for x in mapped_by_cohort.values()]
    )
)
if not common_genes:
    raise RuntimeError("Common-gene intersection is empty.")

X_global = pd.concat(
    [mapped_by_cohort[g][common_genes] for g in DISCOVERY_GSE],
    axis=0,
)
X_global.index = X_global.index.astype(str)
X_global = X_global.loc[locked["GSM_ID"].tolist()]

assert X_global.shape[0] == 608
assert not X_global.isna().any().any()

locked_i = locked.set_index("GSM_ID").loc[X_global.index]
y_global = locked_i["Label"].astype(int)
fold_global = locked_i["Fold"].astype(int)
batch_global = pd.Series(
    [sample_to_cohort[s] for s in X_global.index],
    index=X_global.index,
    name="GEO_ID",
)

pd.DataFrame(log2_rows).to_csv(
    TABLE_DIR / "global_development_log2_audit.csv", index=False
)
pd.DataFrame(selected_probe_rows).to_csv(
    TABLE_DIR / "global_development_representative_probe_mapping.csv",
    index=False,
)
pd.DataFrame({"Gene":common_genes}).to_csv(
    TABLE_DIR / "global_development_common_genes.csv",
    index=False,
)

scope = pd.DataFrame([
    ["Conditional log2","Outside folds","All 608 development",False],
    ["Representative probe","Outside folds","All 608 development",False],
    ["Common-gene intersection","Outside folds","11 discovery cohorts",False],
    ["Quantile reference","Inside fold","Training fold only",False],
    ["ComBat estimates","Inside fold","Training fold only",False],
    ["Bottom-10% variance filter","Inside fold","Training fold only",False],
    ["limma + NIBFS","Inside fold","Training fold only",True],
    ["Classifier fitting","Inside fold","Training fold only",True],
], columns=["Step","Scope","Fitted_using","Uses_class_labels"])
scope.to_csv(
    TABLE_DIR / "preprocessing_scope_design_audit.csv", index=False
)

# Fixed STRING degree
degree_lookup = {}
degree_source = ""

full_rank_path = TABLES_SOURCE_DIR / "full_training_NIBFS_ranking.csv"
if full_rank_path.is_file():
    rank_table = pd.read_csv(full_rank_path)
    gcol = find_column(rank_table, ["Gene","Gene_Symbol","Symbol"])
    dcol = find_column(rank_table, ["Degree","PPI_degree","STRING_degree"])
    if gcol and dcol:
        degree_lookup = (
            rank_table[[gcol,dcol]]
            .drop_duplicates(gcol)
            .assign(**{gcol:rank_table[gcol].astype(str).str.upper().str.strip()})
            .set_index(gcol)[dcol]
            .pipe(pd.to_numeric, errors="coerce")
            .fillna(0)
            .to_dict()
        )
        degree_source = str(full_rank_path)

if not degree_lookup:
    edge_path = TABLES_SOURCE_DIR / "STRING_gene_edges_eligible_genes.csv"
    edges = pd.read_csv(edge_path)
    pairs = [
        ("Gene_A","Gene_B"),("Gene1","Gene2"),("gene1","gene2"),
        ("preferredName_A","preferredName_B"),
    ]
    pair = next(
        (p for p in pairs if p[0] in edges.columns and p[1] in edges.columns),
        None,
    )
    if pair is None:
        raise KeyError(f"Cannot detect STRING endpoint columns: {list(edges.columns)}")
    degree = pd.concat([
        edges[pair[0]].astype(str).str.upper().str.strip(),
        edges[pair[1]].astype(str).str.upper().str.strip(),
    ]).value_counts()
    degree_lookup = degree.astype(float).to_dict()
    degree_source = str(edge_path)

print("Development:", X_global.shape)
print("Cancer/normal:", int(y_global.sum()), int((1-y_global).sum()))
print("Fixed STRING degree genes:", len(degree_lookup))


# =============================================================================
# LOAD THE COMPARATOR IMPLEMENTATIONS FROM THIS REPOSITORY
# =============================================================================
import importlib
import yaml

if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
importlib.invalidate_caches()

from src.workflow import rankings as exact_rankings
from src.feature_selection import select_top_k as exact_select_top_k
from src.modeling import create_models as exact_create_models

cfg = yaml.safe_load((PACKAGE_DIR / "config.yaml").read_text(encoding="utf-8"))
EXPECTED_METHODS = ["NIBFS", "DEG-only", "mRMR", "LASSO"]
configured_methods = list(cfg.get("feature_selection", {}).get("methods", EXPECTED_METHODS))
missing_configured_methods = [method for method in EXPECTED_METHODS if method not in configured_methods]
if missing_configured_methods:
    raise RuntimeError(
        "Required comparator missing from repository config: "
        + ", ".join(missing_configured_methods)
    )

ppi_degree_exact = pd.DataFrame({
    "Gene": list(degree_lookup.keys()),
    "Degree": list(degree_lookup.values()),
})
ppi_degree_exact["Gene"] = ppi_degree_exact["Gene"].astype(str).str.upper().str.strip()
ppi_degree_exact["Degree"] = pd.to_numeric(ppi_degree_exact["Degree"], errors="coerce").fillna(0.0)
ppi_degree_exact = ppi_degree_exact.drop_duplicates("Gene").reset_index(drop=True)
exact_models = exact_create_models(cfg)

print("=" * 88)
print("EXACT COMPARATOR PACKAGE: PASS")
print("=" * 88)
print("Package source :", PACKAGE_DIR)
print("Methods        :", configured_methods)
print("Models         :", list(exact_models.keys()))
print("PPI degree rows:", len(ppi_degree_exact))


In [ ]:

@dataclass
class FrozenQuantileNormalizer:
    reference: np.ndarray | None = None

    def fit(self, X):
        self.reference = np.sort(
            X.to_numpy(dtype=float),
            axis=1,
        ).mean(axis=0)
        return self

    def transform(self, X):
        if self.reference is None:
            raise RuntimeError(
                "Quantile normalizer belum di-fit."
            )

        values = X.to_numpy(dtype=float)
        output = np.empty_like(
            values,
            dtype=float,
        )
        positions = np.arange(
            len(self.reference),
            dtype=float,
        )

        for row_index, row in enumerate(values):
            ranks = rankdata(
                row,
                method="average",
            ) - 1.0

            output[row_index] = np.interp(
                ranks,
                positions,
                self.reference,
            )

        return pd.DataFrame(
            output,
            index=X.index,
            columns=X.columns,
        )

    def fit_transform(self, X):
        return self.fit(X).transform(X)


RANK_COLUMNS = {
    "NIBFS": "Rank_NIBFS",
    "DEG-only": "Rank_stat",
    "mRMR": "Selection_Order",
    "LASSO": "Rank_LASSO",
}


def standardize_method_name(value):
    text = normalize_name(value)
    if text == "nibfs":
        return "NIBFS"
    if text in {"deg", "degonly"}:
        return "DEG-only"
    if "mrmr" in text:
        return "mRMR"
    if "lasso" in text:
        return "LASSO"
    return str(value)


def standardize_classifier_name(value):
    text = normalize_name(value)
    if text in {"lr", "logisticregression", "logistic"}:
        return "LR"
    if text in {"rf", "randomforest"}:
        return "RF"
    if "lightgbm" in text or text == "lgbm":
        return "LightGBM"
    return str(value)


def exact_rankings_for_fold(
    X_train_fold,
    y_train_fold,
):
    ranking_output = exact_rankings(
        X_train_fold,
        np.asarray(
            y_train_fold,
            dtype=int,
        ),
        ppi_degree_exact,
        cfg,
    )

    standardized = {}

    for method_name, ranking_table in ranking_output.items():
        standardized_name = standardize_method_name(
            method_name
        )
        if standardized_name in EXPECTED_METHODS:
            standardized[
                standardized_name
            ] = pd.DataFrame(
                ranking_table
            ).copy()

    missing_methods = [
        method
        for method in EXPECTED_METHODS
        if method not in standardized
    ]

    if missing_methods:
        raise RuntimeError(
            "Exact rankings tidak menghasilkan comparator: "
            + ", ".join(missing_methods)
        )

    for method in EXPECTED_METHODS:
        ranking_table = standardized[method]
        rank_column = RANK_COLUMNS[method]

        if "Gene" not in ranking_table.columns:
            raise KeyError(
                f"{method} ranking tidak memiliki kolom Gene."
            )
        if rank_column not in ranking_table.columns:
            raise KeyError(
                f"{method} ranking tidak memiliki kolom "
                f"{rank_column}. Tersedia: "
                f"{list(ranking_table.columns)}"
            )

        ranking_table["Gene"] = (
            ranking_table["Gene"]
            .astype(str)
            .str.upper()
            .str.strip()
        )
        standardized[method] = ranking_table

    return standardized


def calculate_metrics(
    y_true,
    probability,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )
    probability = np.asarray(
        probability,
        dtype=float,
    )
    predicted = (
        probability >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    ).ravel()

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            probability,
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probability,
        ),
        "Accuracy": accuracy_score(
            y_true,
            predicted,
        ),
        "Balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                predicted,
            ),
        "Sensitivity": recall_score(
            y_true,
            predicted,
            pos_label=1,
            zero_division=0,
        ),
        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "Precision": precision_score(
            y_true,
            predicted,
            zero_division=0,
        ),
        "F1": f1_score(
            y_true,
            predicted,
            zero_division=0,
        ),
        "MCC": matthews_corrcoef(
            y_true,
            predicted,
        ),
        "Brier_score": brier_score_loss(
            y_true,
            probability,
        ),
    }


print("Exact comparator helpers ready.")


In [ ]:

fold_metric_rows = []
prediction_rows = []
panel_rows = []
preprocessing_rows = []
runtime_rows = []

for fold in range(1, N_FOLDS + 1):
    fold_started = time.perf_counter()

    print("\n" + "=" * 88)
    print(
        f"FOLD {fold}/{N_FOLDS} — "
        "NIBFS, DEG-only, mRMR, LASSO"
    )
    print("=" * 88)

    validation_ids = locked.loc[
        locked["Fold"].eq(fold),
        "GSM_ID",
    ].tolist()
    validation_set = set(
        validation_ids
    )

    training_ids = [
        sample_id
        for sample_id in locked["GSM_ID"].tolist()
        if sample_id not in validation_set
    ]

    X_train_raw = X_global.loc[
        training_ids
    ].copy()
    X_valid_raw = X_global.loc[
        validation_ids
    ].copy()

    y_train_fold = y_global.loc[
        training_ids
    ]
    y_valid_fold = y_global.loc[
        validation_ids
    ]

    batch_train = batch_global.loc[
        training_ids
    ]
    batch_valid = batch_global.loc[
        validation_ids
    ]

    if not set(
        batch_valid.unique()
    ).issubset(
        set(batch_train.unique())
    ):
        missing_batches = sorted(
            set(batch_valid.unique())
            - set(batch_train.unique())
        )
        raise RuntimeError(
            f"Fold {fold}: batch validasi tidak ada pada "
            f"training: {missing_batches}"
        )

    # --------------------------------------------------------------
    # A. Quantile reference: fit training fold only
    # --------------------------------------------------------------
    quantile = FrozenQuantileNormalizer()
    X_train_quantile = quantile.fit_transform(
        X_train_raw
    )
    X_valid_quantile = quantile.transform(
        X_valid_raw
    )

    # --------------------------------------------------------------
    # B. Label-free ComBat: fit training fold only
    # --------------------------------------------------------------
    combat_fit = neuroCombat(
        dat=X_train_quantile.T.to_numpy(
            dtype=float
        ),
        covars=pd.DataFrame(
            {
                "batch":
                    batch_train
                    .astype(str)
                    .to_numpy()
            }
        ),
        batch_col="batch",
        categorical_cols=[],
        continuous_cols=[],
        eb=True,
        parametric=True,
        mean_only=False,
    )

    X_train_combat = pd.DataFrame(
        combat_fit["data"].T,
        index=X_train_quantile.index,
        columns=X_train_quantile.columns,
    )

    combat_valid = neuroCombatFromTraining(
        dat=X_valid_quantile.T.to_numpy(
            dtype=float
        ),
        batch=batch_valid.astype(
            str
        ).to_numpy(),
        estimates=combat_fit[
            "estimates"
        ],
    )

    X_valid_combat = pd.DataFrame(
        combat_valid["data"].T,
        index=X_valid_quantile.index,
        columns=X_valid_quantile.columns,
    )

    # --------------------------------------------------------------
    # C. Bottom-10% variance filter: fit training fold only
    # --------------------------------------------------------------
    training_variance = (
        X_train_combat.var(
            axis=0,
            ddof=1,
        )
    )
    variance_cutoff = float(
        training_variance.quantile(
            BOTTOM_VARIANCE_FRACTION
        )
    )
    kept_genes = training_variance[
        training_variance
        > variance_cutoff
    ].index.tolist()

    X_train_fold = X_train_combat[
        kept_genes
    ].copy()
    X_valid_fold = X_valid_combat[
        kept_genes
    ].copy()

    # --------------------------------------------------------------
    # D. Exact original feature selectors: training fold only
    # --------------------------------------------------------------
    print(
        f"[Fold {fold}] Menjalankan exact rankings "
        "NIBFS, DEG-only, mRMR, dan LASSO..."
    )

    ranking_tables = exact_rankings_for_fold(
        X_train_fold,
        y_train_fold,
    )

    selected_panels = {}

    for method in EXPECTED_METHODS:
        ranking_table = (
            ranking_tables[method]
        )
        rank_column = (
            RANK_COLUMNS[method]
        )

        genes = exact_select_top_k(
            ranking_table,
            FINAL_K,
            rank_column,
        )

        genes = list(
            map(str, genes)
        )

        if len(genes) != FINAL_K:
            raise RuntimeError(
                f"Fold {fold}, {method}: "
                f"hanya {len(genes)} gen terpilih."
            )

        if not set(genes).issubset(
            set(X_train_fold.columns)
        ):
            missing_panel = sorted(
                set(genes)
                - set(X_train_fold.columns)
            )
            raise RuntimeError(
                f"Fold {fold}, {method}: gen panel "
                f"tidak tersedia: {missing_panel}"
            )

        selected_panels[
            method
        ] = genes

        ranking_table.to_csv(
            FOLD_DIR
            / (
                f"fold_{fold}_"
                f"{method.replace('-', '_')}_ranking.csv.gz"
            ),
            index=False,
            compression="gzip",
        )

        for selection_rank, gene in enumerate(
            genes,
            start=1,
        ):
            panel_rows.append(
                {
                    "Fold": fold,
                    "Method": method,
                    "k": FINAL_K,
                    "Selection_rank":
                        selection_rank,
                    "Gene": gene,
                }
            )

    # --------------------------------------------------------------
    # E. Same classifiers for every selected panel
    # --------------------------------------------------------------
    for method in EXPECTED_METHODS:
        panel_genes = (
            selected_panels[method]
        )

        for raw_model_name, estimator in exact_models.items():
            classifier = standardize_classifier_name(
                raw_model_name
            )

            fitted = clone(
                estimator
            ).fit(
                X_train_fold[
                    panel_genes
                ],
                y_train_fold,
            )

            probability = (
                fitted.predict_proba(
                    X_valid_fold[
                        panel_genes
                    ]
                )[:, 1]
            )

            metrics = calculate_metrics(
                y_valid_fold,
                probability,
            )

            fold_metric_rows.append(
                {
                    "Fold": fold,
                    "Method": method,
                    "Classifier":
                        classifier,
                    "k": FINAL_K,
                    **metrics,
                }
            )

            for sample_id, truth, score in zip(
                validation_ids,
                y_valid_fold.loc[
                    validation_ids
                ].to_numpy(),
                probability,
            ):
                prediction_rows.append(
                    {
                        "Fold": fold,
                        "Method": method,
                        "Classifier":
                            classifier,
                        "k": FINAL_K,
                        "Sample_ID":
                            sample_id,
                        "True_Label":
                            int(truth),
                        "Probability":
                            float(score),
                        "GEO_ID":
                            batch_global.loc[
                                sample_id
                            ],
                    }
                )

    preprocessing_rows.append(
        {
            "Fold": fold,
            "Training_samples":
                len(training_ids),
            "Validation_samples":
                len(validation_ids),
            "Training_batches":
                int(batch_train.nunique()),
            "Validation_batches":
                int(batch_valid.nunique()),
            "Common_genes_before_fold_preprocessing":
                X_train_raw.shape[1],
            "Genes_after_variance_filter":
                X_train_fold.shape[1],
            "Variance_cutoff":
                variance_cutoff,
            "Methods_compared":
                ";".join(
                    EXPECTED_METHODS
                ),
            "Validation_used_to_fit_quantile_reference":
                False,
            "Validation_used_to_fit_ComBat":
                False,
            "Validation_used_to_fit_variance_filter":
                False,
            "Validation_used_for_feature_selection":
                False,
        }
    )

    elapsed = (
        time.perf_counter()
        - fold_started
    )
    runtime_rows.append(
        {
            "Fold": fold,
            "Runtime_seconds":
                elapsed,
            "Runtime_minutes":
                elapsed / 60.0,
        }
    )

    # Fold checkpoint
    pd.DataFrame(
        fold_metric_rows
    ).to_csv(
        TABLE_DIR
        / "all_comparators_fold_metrics_checkpoint.csv",
        index=False,
    )
    pd.DataFrame(
        prediction_rows
    ).to_csv(
        TABLE_DIR
        / "all_comparators_oof_predictions_checkpoint.csv",
        index=False,
    )
    pd.DataFrame(
        panel_rows
    ).to_csv(
        TABLE_DIR
        / "all_comparators_selected_panels_checkpoint.csv",
        index=False,
    )

    print(
        f"Fold {fold} selesai dalam "
        f"{elapsed / 60:.2f} menit."
    )

    display(
        pd.DataFrame(
            fold_metric_rows
        ).query(
            "Fold == @fold"
        )[
            [
                "Method",
                "Classifier",
                "ROC_AUC",
                "Balanced_accuracy",
                "F1",
                "MCC",
            ]
        ].sort_values(
            [
                "Method",
                "Classifier",
            ]
        )
    )

fold_metrics = pd.DataFrame(
    fold_metric_rows
)
oof_predictions = pd.DataFrame(
    prediction_rows
)
selected_panels = pd.DataFrame(
    panel_rows
)
preprocessing_audit = pd.DataFrame(
    preprocessing_rows
)
runtime_table = pd.DataFrame(
    runtime_rows
)

fold_metrics.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_fold_metrics.csv",
    index=False,
)
oof_predictions.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_oof_predictions.csv",
    index=False,
)
selected_panels.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_selected_panels.csv",
    index=False,
)
preprocessing_audit.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_preprocessing_audit.csv",
    index=False,
)
runtime_table.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_runtime.csv",
    index=False,
)

print(
    "All five folds and four methods completed."
)


In [ ]:

from scipy.stats import (
    friedmanchisquare,
    wilcoxon,
)
from statsmodels.stats.multitest import (
    multipletests,
)

metric_names = [
    "ROC_AUC",
    "PR_AUC",
    "Accuracy",
    "Balanced_accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "MCC",
    "Brier_score",
]

performance_rows = []

for (
    method,
    classifier,
), block in fold_metrics.groupby(
    [
        "Method",
        "Classifier",
    ]
):
    row = {
        "Method": method,
        "Classifier": classifier,
        "k": FINAL_K,
        "Folds": int(
            block["Fold"].nunique()
        ),
    }

    for metric in metric_names:
        values = pd.to_numeric(
            block[metric],
            errors="coerce",
        )

        row[
            f"{metric}_mean"
        ] = float(
            values.mean()
        )
        row[
            f"{metric}_sd"
        ] = float(
            values.std(ddof=1)
        )
        row[
            f"{metric}_min"
        ] = float(
            values.min()
        )
        row[
            f"{metric}_max"
        ] = float(
            values.max()
        )

    performance_rows.append(
        row
    )

performance_summary = pd.DataFrame(
    performance_rows
).sort_values(
    [
        "Method",
        "Classifier",
    ]
)

performance_summary.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_1x5_performance_summary.csv",
    index=False,
)

# ------------------------------------------------------------------
# Stability for every method
# ------------------------------------------------------------------
pairwise_rows = []
frequency_rows = []
stability_rows = []

for method in EXPECTED_METHODS:
    method_panels = selected_panels[
        selected_panels[
            "Method"
        ].eq(method)
    ].copy()

    fold_sets = {
        int(fold): set(
            block.sort_values(
                "Selection_rank"
            )["Gene"].head(
                FINAL_K
            )
        )
        for fold, block
        in method_panels.groupby(
            "Fold"
        )
    }

    for fold_a, fold_b in combinations(
        sorted(fold_sets),
        2,
    ):
        left = fold_sets[fold_a]
        right = fold_sets[fold_b]
        union = left | right

        pairwise_rows.append(
            {
                "Method": method,
                "Fold_A": fold_a,
                "Fold_B": fold_b,
                "Intersection":
                    len(left & right),
                "Union":
                    len(union),
                "Jaccard":
                    (
                        len(left & right)
                        / len(union)
                    ),
            }
        )

    counts = (
        method_panels[
            [
                "Fold",
                "Gene",
            ]
        ]
        .drop_duplicates()
        .groupby(
            "Gene"
        )["Fold"]
        .nunique()
        .sort_values(
            ascending=False
        )
    )

    for gene, frequency in counts.items():
        frequency_rows.append(
            {
                "Method":
                    method,
                "Gene":
                    gene,
                "Fold_frequency":
                    int(frequency),
            }
        )

    method_pair_values = [
        row["Jaccard"]
        for row in pairwise_rows
        if row["Method"] == method
    ]

    stability_rows.append(
        {
            "Method": method,
            "k": FINAL_K,
            "Mean_Jaccard":
                float(
                    np.mean(
                        method_pair_values
                    )
                ),
            "SD_Jaccard":
                float(
                    np.std(
                        method_pair_values,
                        ddof=1,
                    )
                ),
            "Min_Jaccard":
                float(
                    np.min(
                        method_pair_values
                    )
                ),
            "Max_Jaccard":
                float(
                    np.max(
                        method_pair_values
                    )
                ),
            "Genes_in_all_5_folds":
                int(
                    (counts == 5).sum()
                ),
            "Genes_in_at_least_4_folds":
                int(
                    (counts >= 4).sum()
                ),
        }
    )

pairwise_jaccard = pd.DataFrame(
    pairwise_rows
).sort_values(
    [
        "Method",
        "Fold_A",
        "Fold_B",
    ]
)

gene_frequency = pd.DataFrame(
    frequency_rows
).sort_values(
    [
        "Method",
        "Fold_frequency",
        "Gene",
    ],
    ascending=[
        True,
        False,
        True,
    ],
)

stability_summary = pd.DataFrame(
    stability_rows
).sort_values(
    "Mean_Jaccard",
    ascending=False,
)

pairwise_jaccard.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_pairwise_jaccard.csv",
    index=False,
)
gene_frequency.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_gene_frequency.csv",
    index=False,
)
stability_summary.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_stability_summary.csv",
    index=False,
)

# ------------------------------------------------------------------
# Paired statistical tests for stability
# ------------------------------------------------------------------
stability_wide = (
    pairwise_jaccard.assign(
        Fold_pair=lambda frame:
            frame["Fold_A"].astype(str)
            + "-"
            + frame["Fold_B"].astype(str)
    )
    .pivot(
        index="Fold_pair",
        columns="Method",
        values="Jaccard",
    )
    .dropna()
)

friedman_statistic, friedman_p = (
    friedmanchisquare(
        *[
            stability_wide[
                method
            ].to_numpy()
            for method in EXPECTED_METHODS
        ]
    )
)

stability_friedman = pd.DataFrame(
    [
        {
            "Test":
                "Friedman paired test",
            "Outcome":
                "Pairwise Jaccard",
            "Methods":
                ";".join(
                    EXPECTED_METHODS
                ),
            "N_paired_fold_pairs":
                len(
                    stability_wide
                ),
            "Statistic":
                float(
                    friedman_statistic
                ),
            "P_value":
                float(
                    friedman_p
                ),
        }
    ]
)

stability_posthoc_rows = []

for method_a, method_b in combinations(
    EXPECTED_METHODS,
    2,
):
    statistic, p_value = wilcoxon(
        stability_wide[
            method_a
        ],
        stability_wide[
            method_b
        ],
        zero_method="wilcox",
        alternative="two-sided",
    )

    stability_posthoc_rows.append(
        {
            "Method_A": method_a,
            "Method_B": method_b,
            "Wilcoxon_statistic":
                float(statistic),
            "P_value":
                float(p_value),
            "Mean_Jaccard_A":
                float(
                    stability_wide[
                        method_a
                    ].mean()
                ),
            "Mean_Jaccard_B":
                float(
                    stability_wide[
                        method_b
                    ].mean()
                ),
        }
    )

stability_posthoc = pd.DataFrame(
    stability_posthoc_rows
)

stability_posthoc[
    "P_adjusted_BH"
] = multipletests(
    stability_posthoc[
        "P_value"
    ].to_numpy(),
    method="fdr_bh",
)[1]

stability_posthoc[
    "Significant_FDR_0.05"
] = (
    stability_posthoc[
        "P_adjusted_BH"
    ] < 0.05
)

stability_friedman.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_stability_Friedman.csv",
    index=False,
)
stability_posthoc.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_stability_Wilcoxon_BH.csv",
    index=False,
)

# ------------------------------------------------------------------
# Paired foldwise ROC-AUC comparisons within each classifier
# ------------------------------------------------------------------
auc_posthoc_rows = []

for classifier in sorted(
    fold_metrics[
        "Classifier"
    ].unique()
):
    classifier_auc = (
        fold_metrics[
            fold_metrics[
                "Classifier"
            ].eq(classifier)
        ]
        .pivot(
            index="Fold",
            columns="Method",
            values="ROC_AUC",
        )
        .dropna()
    )

    for method_a, method_b in combinations(
        EXPECTED_METHODS,
        2,
    ):
        statistic, p_value = wilcoxon(
            classifier_auc[
                method_a
            ],
            classifier_auc[
                method_b
            ],
            zero_method="wilcox",
            alternative="two-sided",
        )

        auc_posthoc_rows.append(
            {
                "Classifier":
                    classifier,
                "Method_A":
                    method_a,
                "Method_B":
                    method_b,
                "Mean_ROC_AUC_A":
                    float(
                        classifier_auc[
                            method_a
                        ].mean()
                    ),
                "Mean_ROC_AUC_B":
                    float(
                        classifier_auc[
                            method_b
                        ].mean()
                    ),
                "Mean_difference_A_minus_B":
                    float(
                        (
                            classifier_auc[
                                method_a
                            ]
                            - classifier_auc[
                                method_b
                            ]
                        ).mean()
                    ),
                "Wilcoxon_statistic":
                    float(
                        statistic
                    ),
                "P_value":
                    float(
                        p_value
                    ),
            }
        )

auc_posthoc = pd.DataFrame(
    auc_posthoc_rows
)

auc_posthoc[
    "P_adjusted_BH"
] = multipletests(
    auc_posthoc[
        "P_value"
    ].to_numpy(),
    method="fdr_bh",
)[1]

auc_posthoc[
    "Significant_FDR_0.05"
] = (
    auc_posthoc[
        "P_adjusted_BH"
    ] < 0.05
)

auc_posthoc.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_ROCAUC_Wilcoxon_BH.csv",
    index=False,
)

# ------------------------------------------------------------------
# Compare every method/classifier with the main pipeline
# ------------------------------------------------------------------
main_performance = pd.read_csv(
    TABLES_SOURCE_DIR
    / "cross_validated_performance_summary_k20.csv"
)

main_method_col = find_column(
    main_performance,
    [
        "Feature_selection_method",
        "Method",
    ],
)
main_classifier_col = find_column(
    main_performance,
    [
        "Classifier",
        "Model",
    ],
)
main_auc_col = find_column(
    main_performance,
    [
        "ROC_AUC_mean",
        "AUC_mean",
    ],
)

main_table = pd.DataFrame(
    {
        "Method":
            main_performance[
                main_method_col
            ].map(
                standardize_method_name
            ),
        "Classifier":
            main_performance[
                main_classifier_col
            ].map(
                standardize_classifier_name
            ),
        "Main_mean_ROC_AUC":
            pd.to_numeric(
                main_performance[
                    main_auc_col
                ],
                errors="coerce",
            ),
    }
)

main_table = (
    main_table[
        main_table[
            "Method"
        ].isin(
            EXPECTED_METHODS
        )
    ]
    .dropna(
        subset=[
            "Main_mean_ROC_AUC"
        ]
    )
    .drop_duplicates(
        [
            "Method",
            "Classifier",
        ]
    )
)

auc_comparison = (
    performance_summary[
        [
            "Method",
            "Classifier",
            "ROC_AUC_mean",
            "ROC_AUC_sd",
        ]
    ]
    .rename(
        columns={
            "ROC_AUC_mean":
                "Fold_fitted_mean_ROC_AUC",
            "ROC_AUC_sd":
                "Fold_fitted_SD_ROC_AUC",
        }
    )
    .merge(
        main_table,
        on=[
            "Method",
            "Classifier",
        ],
        how="left",
    )
)

auc_comparison[
    "Delta_fold_fitted_minus_main"
] = (
    auc_comparison[
        "Fold_fitted_mean_ROC_AUC"
    ]
    - auc_comparison[
        "Main_mean_ROC_AUC"
    ]
)

auc_comparison[
    "Absolute_delta"
] = (
    auc_comparison[
        "Delta_fold_fitted_minus_main"
    ].abs()
)

auc_comparison[
    "Within_prespecified_margin_0.02"
] = (
    auc_comparison[
        "Absolute_delta"
    ] <= AUC_MARGIN
)

auc_comparison.to_csv(
    TABLE_DIR
    / "fold_fitted_all_methods_vs_main_auc.csv",
    index=False,
)

# ------------------------------------------------------------------
# Figures
# ------------------------------------------------------------------
method_order = EXPECTED_METHODS
classifier_order = [
    "LR",
    "RF",
    "LightGBM",
]

fig, ax = plt.subplots(
    figsize=(12, 6)
)

x = np.arange(
    len(method_order)
)
bar_width = 0.24

for classifier_index, classifier in enumerate(
    classifier_order
):
    plot_block = (
        performance_summary[
            performance_summary[
                "Classifier"
            ].eq(classifier)
        ]
        .set_index(
            "Method"
        )
        .reindex(
            method_order
        )
    )

    ax.bar(
        x
        + (
            classifier_index - 1
        ) * bar_width,
        plot_block[
            "ROC_AUC_mean"
        ],
        bar_width,
        label=classifier,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    method_order
)
ax.set_ylim(
    0.5,
    1.01,
)
ax.set_ylabel(
    "Mean five-fold ROC-AUC"
)
ax.set_title(
    "Fold-fitted preprocessing: "
    "feature-selection comparator performance"
)
ax.legend()
fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "Figure_fold_fitted_all_methods_ROCAUC.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR
    / "Figure_fold_fitted_all_methods_ROCAUC.pdf",
    bbox_inches="tight",
)
plt.close(fig)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

stability_plot = (
    stability_summary
    .set_index(
        "Method"
    )
    .reindex(
        method_order
    )
)

ax.bar(
    method_order,
    stability_plot[
        "Mean_Jaccard"
    ],
)

ax.errorbar(
    method_order,
    stability_plot[
        "Mean_Jaccard"
    ],
    yerr=stability_plot[
        "SD_Jaccard"
    ],
    fmt="none",
    capsize=4,
)

ax.set_ylim(
    0,
    1.02,
)
ax.set_ylabel(
    "Mean pairwise Jaccard"
)
ax.set_title(
    "Fold-fitted preprocessing: "
    "panel stability comparison"
)
fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "Figure_fold_fitted_all_methods_stability.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR
    / "Figure_fold_fitted_all_methods_stability.pdf",
    bbox_inches="tight",
)
plt.close(fig)

# ------------------------------------------------------------------
# Supplementary table and manuscript text
# ------------------------------------------------------------------
supplementary_table = (
    performance_summary.merge(
        stability_summary[
            [
                "Method",
                "Mean_Jaccard",
                "SD_Jaccard",
                "Genes_in_all_5_folds",
            ]
        ],
        on="Method",
        how="left",
    )
)

supplementary_table.to_csv(
    TABLE_DIR
    / "Supplementary_Table_fold_fitted_all_comparators_1x5.csv",
    index=False,
)

nibfs_stability = stability_summary.loc[
    stability_summary[
        "Method"
    ].eq(
        "NIBFS"
    ),
    "Mean_Jaccard",
].iloc[0]

best_stability_row = stability_summary.iloc[0]
best_auc_row = (
    performance_summary.sort_values(
        "ROC_AUC_mean",
        ascending=False,
    ).iloc[0]
)

manuscript_text = f"""SUPPLEMENTARY METHODS

The fold-fitted sensitivity analysis was repeated for all four feature-selection
methods used in the main study: NIBFS, DEG-only, mRMR, and LASSO. All methods
used the same locked five folds and identical fold-training preprocessing.
Within each fold, the quantile-normalization reference, label-free ComBat
parameters, bottom-10% variance filter, feature ranking, and classifier fitting
were estimated from training samples only. The held-out fold was transformed
with frozen training-fitted objects. The 152 locked internal-test samples,
GSE15852, GSE70947, and TCGA-BRCA were excluded.

SUPPLEMENTARY RESULTS

The highest mean panel stability was obtained by
{best_stability_row['Method']} (mean Jaccard
{best_stability_row['Mean_Jaccard']:.4f}). NIBFS achieved a mean Jaccard of
{nibfs_stability:.4f}. The highest mean predictive discrimination across all
method-classifier combinations was obtained by
{best_auc_row['Method']} with {best_auc_row['Classifier']}
(ROC-AUC {best_auc_row['ROC_AUC_mean']:.4f}). Formal paired Friedman and
Wilcoxon tests are reported in the accompanying supplementary tables.

INTERPRETATION

This analysis provides the direct apples-to-apples comparator assessment under
the stricter fold-fitted preprocessing protocol. It supersedes the earlier
NIBFS-only fold-fitted sensitivity result for claims involving comparisons
among feature-selection methods, but it does not replace the frozen main panel
or the three independent external validations.
"""

(
    LOCAL_OUTPUT_DIR
    / "manuscript_text_fold_fitted_all_comparators_1x5.txt"
).write_text(
    manuscript_text,
    encoding="utf-8",
)

# ------------------------------------------------------------------
# Audit and ZIP
# ------------------------------------------------------------------
summary = {
    "completed_utc":
        utc_now(),
    "status":
        "PASS",
    "analysis":
        (
            "1x5 fold-fitted harmonization "
            "with NIBFS, DEG-only, mRMR, and LASSO"
        ),
    "methods":
        EXPECTED_METHODS,
    "classifiers":
        sorted(
            fold_metrics[
                "Classifier"
            ].unique().tolist()
        ),
    "development_samples":
        int(
            len(locked)
        ),
    "development_cancer":
        int(
            locked[
                "Label"
            ].sum()
        ),
    "development_normal":
        int(
            (
                1
                - locked[
                    "Label"
                ]
            ).sum()
        ),
    "common_genes_before_fold_preprocessing":
        int(
            len(common_genes)
        ),
    "external_geo_excluded":
        EXTERNAL_GEO,
    "external_rnaseq_excluded":
        EXTERNAL_RNASEQ,
    "stability_friedman_p":
        float(
            friedman_p
        ),
    "best_stability_method":
        str(
            best_stability_row[
                "Method"
            ]
        ),
    "best_stability_mean_jaccard":
        float(
            best_stability_row[
                "Mean_Jaccard"
            ]
        ),
    "best_auc_method":
        str(
            best_auc_row[
                "Method"
            ]
        ),
    "best_auc_classifier":
        str(
            best_auc_row[
                "Classifier"
            ]
        ),
    "best_auc_mean":
        float(
            best_auc_row[
                "ROC_AUC_mean"
            ]
        ),
    "scientific_scope":
        (
            "Direct comparator sensitivity under identical "
            "fold-fitted preprocessing."
        ),
}

(
    LOCAL_OUTPUT_DIR
    / "fold_fitted_all_comparators_audit_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

manifest_rows = []

for path in sorted(
    LOCAL_OUTPUT_DIR.rglob(
        "*"
    )
):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path":
                    str(
                        path.relative_to(
                            LOCAL_OUTPUT_DIR
                        )
                    ),
                "bytes":
                    path.stat().st_size,
                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )

pd.DataFrame(
    manifest_rows
).to_csv(
    LOCAL_OUTPUT_DIR
    / "fold_fitted_all_comparators_file_manifest.csv",
    index=False,
)

zip_path = Path(
    shutil.make_archive(
        str(
            LOCAL_OUTPUT_DIR
        ),
        "zip",
        root_dir=LOCAL_OUTPUT_DIR.parent,
        base_dir=LOCAL_OUTPUT_DIR.name,
    )
)

archive_dir = PACKAGE_DIR / "results" / "archives"
archive_dir.mkdir(parents=True, exist_ok=True)
drive_zip_path = archive_dir / "FOLD_FITTED_ALL_COMPARATORS_FROM_SERIES_MATRIX_1X5.zip"
if zip_path.resolve() != drive_zip_path.resolve():
    shutil.copy2(zip_path, drive_zip_path)
drive_copy_status = "success"

print("\n" + "=" * 88)
print(
    "FOLD-FITTED ALL-COMPARATOR ANALYSIS COMPLETE"
)
print("=" * 88)
print(
    "Methods:",
    EXPECTED_METHODS,
)
print(
    "Local ZIP:",
    zip_path,
)
print(
    "Archive copy:",
    drive_copy_status,
)

if drive_copy_status == "success":
    print(
        "Archive ZIP:",
        drive_zip_path,
    )
else:
    print(
        "Unduh ZIP lokal melalui panel Files Colab."
    )

print("\nPerformance summary:")
display(
    performance_summary
)

print("\nStability summary:")
display(
    stability_summary
)

print("\nStability Friedman:")
display(
    stability_friedman
)

print("\nStability pairwise Wilcoxon-BH:")
display(
    stability_posthoc
)

print("\nROC-AUC pairwise Wilcoxon-BH:")
display(
    auc_posthoc
)
